# Crack the latent baseline

**Target**: beat `heavy_spike_only_within_p8` (within-only) on spatial R².
**Starting point**: `heavy_spike_spike_dom_p8` (cross-modal, same encoder).

Experiments vary **projection head** and **proj_dim** — the hypothesis being that a non-linear head
lets alignment happen in projection space while keeping the encoder embeddings spatially organised.

Evaluation: **Z_spike**, **Z_lfp**, and **Z_concat** R² reported separately every eval.

In [ ]:
import pickle, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

import sys
for _p in [Path('.'), *Path('.').resolve().parents]:
    if (_p / 'comets' / '__init__.py').exists():
        if str(_p) not in sys.path: sys.path.insert(0, str(_p))
        break
from comets import (
    ModalityConfig, SamplingConfig, LatentTrainingConfig,
    CEBRASKipEncoder, SkippedConvEncoderConfig,
    LinearHead, OneLayerMLP, TwoLayerMLP,
    train_latent,
)

ROOT     = Path('.').resolve()
DATA_DIR = '/mnt/d/comets/data/crcns_mouse'
SESSION  = 'ec013.156'

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

def _style_map(styler, fn, **kw):
    try: return styler.map(fn, **kw)
    except AttributeError: return styler.applymap(fn, **kw)

print('Setup OK')

In [ ]:
data = pickle.load(open(f'{DATA_DIR}/{SESSION}/{SESSION}_lfp_spike.pkl', 'rb'))
lfp        = np.asarray(data['lfp_theta_50hz'], dtype=np.float32)
spike_rate = np.asarray(data['spike_rate'],     dtype=np.float32)
position   = np.asarray(data['position_50hz'],  dtype=np.float64)
if position.ndim == 2: position = position[:, 0]

rmin, rmax = spike_rate.min(0, keepdims=True), spike_rate.max(0, keepdims=True)
spike_norm = (spike_rate - rmin) / (rmax - rmin + 1e-9)

T, N = spike_norm.shape
C    = lfp.shape[1]
print(f'Session {SESSION}:  T={T}  N={N} neurons  C={C} LFP channels')

In [ ]:
def window_kfold_r2(Z, pos=position, win_size=10, n_folds=10, alpha=1.0):
    Z   = np.asarray(Z,   dtype=np.float64)
    pos = np.asarray(pos, dtype=np.float64)
    T_  = min(len(Z), len(pos))
    nw  = T_ // win_size
    Z_w = Z[:nw*win_size].reshape(nw, win_size, -1).mean(1)
    y_w = pos[:nw*win_size].reshape(nw, win_size).mean(1)
    fs  = nw // n_folds;  n = fs * n_folds
    Z_w, y_w = Z_w[:n], y_w[:n]
    r2s = []
    for k in range(n_folds):
        mask = np.zeros(n, dtype=bool);  mask[k*fs:(k+1)*fs] = True
        reg = Ridge(alpha=alpha).fit(Z_w[~mask], y_w[~mask])
        r2s.append(float(r2_score(y_w[mask], reg.predict(Z_w[mask]))))
    return float(np.mean(r2s))

def eval_embeddings(Z_spk, Z_lf):
    """Returns dict: r2_spike, r2_lfp, r2_concat."""
    return dict(
        r2_spike  = window_kfold_r2(Z_spk),
        r2_lfp    = window_kfold_r2(Z_lf),
        r2_concat = window_kfold_r2(np.concatenate([Z_spk, Z_lf], axis=1)),
    )

print('Eval helpers ready.')

---
## Baselines (loaded from grid runs, no retraining)

In [ ]:
def load_run(run_name, ckp='best'):
    d = ROOT / 'runs' / 'latent' / run_name / SESSION
    Z_spk = np.load(d / f'Z_spikes_{ckp}.npy')
    Z_lf  = np.load(d / f'Z_lfp_{ckp}.npy')
    return eval_embeddings(Z_spk, Z_lf)

SCORES = {}
SCORES['only_within (target)'] = load_run('heavy_spike_only_within_p8')
SCORES['spike_dom  (start)']   = load_run('heavy_spike_spike_dom_p8')

TARGET = SCORES['only_within (target)']['r2_concat']   # the number to beat

for name, s in SCORES.items():
    print(f'{name:30s}  spike={s["r2_spike"]:+.4f}  lfp={s["r2_lfp"]:+.4f}  concat={s["r2_concat"]:+.4f}')

---
## quick_train

In [ ]:
def quick_train(
    label,
    alpha_cross = 0.1,
    beta        = 0.95,
    head_cls    = TwoLayerMLP,   # LinearHead | OneLayerMLP | TwoLayerMLP
    proj_dim    = 8,
    temperature = 1.0,
    epochs      = 200,
    eval_every  = 20,
    lr          = 1e-3,
    batch_size  = 256,
    # encoder — defaults to heavy_spike (matches grid baseline)
    emb_A=32, hidden_A=64, blocks_A=4, win_A=8,
    emb_B=16, hidden_B=32, blocks_B=2, win_B=8,
):
    modA = ModalityConfig(CEBRASKipEncoder,
                          SkippedConvEncoderConfig(N, emb_A, hidden_A, blocks_A), win_len=win_A)
    modB = ModalityConfig(CEBRASKipEncoder,
                          SkippedConvEncoderConfig(C, emb_B, hidden_B, blocks_B), win_len=win_B)
    sampling = SamplingConfig(batch_size=batch_size, pos_offset=5, negative_mask_dist=16)
    cfg      = LatentTrainingConfig(
        epochs=epochs, lr=lr, temperature=temperature,
        proj_dim=proj_dim, alpha_cross=alpha_cross,
        beta=beta, head_cls=head_cls,
    )

    log = dict(epoch=[], r2_spike=[], r2_lfp=[], r2_concat=[])

    def cb(ep, model):
        Z_s = model.embed_latent_spikes(spike_norm)
        Z_l = model.embed_latent_lfp(lfp)
        s = eval_embeddings(Z_s, Z_l)
        for k in log: log[k].append(ep if k=='epoch' else s[k])
        beat = ' ← BEAT!' if s['r2_concat'] > TARGET else ''
        print(f'  ep {ep:3d}  spike={s["r2_spike"]:+.4f}  lfp={s["r2_lfp"]:+.4f}  concat={s["r2_concat"]:+.4f}{beat}')

    model = train_latent(spike_norm, modA, lfp, modB, sampling, cfg,
                         epoch_callback=cb, eval_every=eval_every)

    Z_spk = model.embed_latent_spikes(spike_norm)
    Z_lf  = model.embed_latent_lfp(lfp)
    scores = eval_embeddings(Z_spk, Z_lf)
    SCORES[label] = scores

    fig, axes = plt.subplots(1, 3, figsize=(13, 3))
    for ax, key, color in [
        (axes[0], 'r2_spike',  'tab:blue'),
        (axes[1], 'r2_lfp',    'tab:green'),
        (axes[2], 'r2_concat', 'tab:orange'),
    ]:
        ax.plot(log['epoch'], log[key], color=color, lw=1.5)
        ax.axhline(SCORES['only_within (target)'][key], color='gray', ls='--', lw=1, label='only_within')
        ax.axhline(SCORES['spike_dom  (start)'][key],  color='salmon', ls=':', lw=1, label='spike_dom')
        ax.set(xlabel='Epoch', title=key); ax.legend(fontsize=7); ax.grid(alpha=0.3)
    fig.suptitle(label, fontweight='bold'); plt.tight_layout(); plt.show()

    print(f'  FINAL  spike={scores["r2_spike"]:+.4f}  lfp={scores["r2_lfp"]:+.4f}  concat={scores["r2_concat"]:+.4f}  (Δ concat vs only_within: {scores["r2_concat"]-TARGET:+.4f})')
    return model, scores

print('quick_train ready.')

---
## Experiments — spike_dom + head / proj_dim variations

Fixed: `heavy_spike` encoder, `alpha_cross=0.1`, `beta=0.95`.  
Varied: `head_cls`, `proj_dim`, `temperature`.

In [ ]:
# TwoLayerMLP head, proj_dim=8  (same dim as grid, different head)
model_1, s1 = quick_train('spike_dom  TwoLayerMLP  p8',
    head_cls=TwoLayerMLP, proj_dim=8)

In [ ]:
# TwoLayerMLP head, proj_dim=16
model_2, s2 = quick_train('spike_dom  TwoLayerMLP  p16',
    head_cls=TwoLayerMLP, proj_dim=16)

In [ ]:
# TwoLayerMLP head, proj_dim=32
model_3, s3 = quick_train('spike_dom  TwoLayerMLP  p32',
    head_cls=TwoLayerMLP, proj_dim=32)

In [ ]:
# OneLayerMLP head, proj_dim=8
model_4, s4 = quick_train('spike_dom  OneLayerMLP  p8',
    head_cls=OneLayerMLP, proj_dim=8)

In [ ]:
# TwoLayerMLP, sharper temperature
model_5, s5 = quick_train('spike_dom  TwoLayerMLP  p8  τ=0.5',
    head_cls=TwoLayerMLP, proj_dim=8, temperature=0.5)

In [ ]:
# Free cell — your config here
model_X, sX = quick_train('my_experiment',
    head_cls=TwoLayerMLP, proj_dim=8,
    alpha_cross=0.1, beta=0.95,
)

---
## Summary table

In [ ]:
df = pd.DataFrame(SCORES).T
target_row = SCORES['only_within (target)']

def color(val, col):
    base = target_row[col]
    d = val - base
    if d > 0.005:  return 'background-color: rgba(0,180,0,0.4)'
    if d < -0.005: return 'background-color: rgba(200,0,0,0.25)'
    return ''

s = df.style.format('{:+.4f}')
for col in df.columns:
    s = _style_map(s, lambda v, c=col: color(v, c), subset=[col])
display(s.set_caption('Green = beats only_within baseline on that metric'))

---
## Unit embedding (optional — run after finding a winning latent config)

Uncomment and set `WINNING_MODEL` to the model you want to evaluate.

In [ ]:
# WINNING_MODEL = model_1   # <-- set to whichever model won
# EMBEDDING_LABEL = 'spike_dom_TwoLayerMLP_p8'
#
# import pickle as _pkl
# from scipy.ndimage import gaussian_filter1d
# from comets import train_unit, UnitTrainingConfig, SamplingConfig as SC
# from comets.unit_eval import ratemap_correlation
#
# spike_count = np.asarray(
#     _pkl.load(open(f'{DATA_DIR}/{SESSION}/{SESSION}_lfp_spike.pkl','rb'))['spike_bin'],
#     dtype=np.float32)
#
# Z_spk_win = WINNING_MODEL.embed_latent_spikes(spike_norm)
# Z_lf_win  = WINNING_MODEL.embed_latent_lfp(lfp)
#
# # ratemaps
# edges    = np.linspace(position.min(), position.max(), 41)
# pos_bins = np.clip(np.digitize(position, edges)-1, 0, 39)
# fs       = float(data['fs_common'])
# occ      = np.bincount(pos_bins, minlength=40) / fs
# rm = np.zeros((spike_count.shape[1], 40), dtype=np.float32)
# for i in range(spike_count.shape[1]):
#     rm[i] = np.bincount(pos_bins, weights=spike_count[:,i], minlength=40) / (occ+1e-9)
# from scipy.ndimage import gaussian_filter1d
# rm = gaussian_filter1d(rm, sigma=2.0, axis=1)
#
# unit_enc = train_unit(None, spike_count, SC(batch_size=256),
#                       UnitTrainingConfig(epochs=200, d_u=8, lambda_coact=0.25, lambda_ctx=0.75),
#                       z_spikes=Z_spk_win, z_lfp=Z_lf_win)
# rho, _ = ratemap_correlation(unit_enc.embeddings().numpy(), rm)
# print(f'{EMBEDDING_LABEL}  rho_ratemap = {rho:+.4f}')